# Cross-Validation

**Topic:** Reliable Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** why a single train/test split gives an unreliable estimate of model performance
- **Describe** how k-fold cross-validation uses the full dataset for both training and validation
- **Interpret** the mean and standard deviation of cross-validation scores

> **Tip:** Start with k=2 and increase the slider. Watch how the mean score stabilizes and the standard deviation shrinks as k grows. Then switch to **Stratified K-Fold** for classification data and notice how it maintains class balance in each fold.

---
## How we got here

In **05_overfitting_and_underfitting.ipynb** we used a validation set to detect overfitting. But there is a problem: a single validation set might get lucky or unlucky, giving a misleading picture of true performance.

This notebook revisits a concept you already saw in:
- **preprocessing/08_cross_validation_and_preprocessing.ipynb** — we showed why cross-validation must be applied inside the pipeline

Now we go deeper: not just *how* to use cross-validation, but *why* it gives more reliable estimates than a single split.

---
## Why this matters for data science

With a single validation split, your estimate of model performance depends heavily on which 20% of the data ended up in the validation set. On a small dataset, this can swing accuracy estimates by 5 to 10 percentage points.

Cross-validation removes that randomness by systematically rotating which data is used for validation. Every example is used for validation exactly once. The result is a more honest, less variable estimate of how the model will perform on new data.

---
## Try it yourself

In [2]:
np.random.seed(42)
X, y = make_classification(n_samples=100, n_features=2,
    n_informative=2, n_redundant=0, weights=[0.8, 0.2], random_state=42)

out = Output()

k_slider = IntSlider(value=5, min=2, max=10, step=1,
    description="k (folds):",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="380px"))

method_dd = Dropdown(
    options=["K-Fold", "Stratified K-Fold", "Leave-One-Out"],
    value="K-Fold",
    description="Method:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="380px"))

caption = widgets.HTML(layout=widgets.Layout(width="640px"))

TRAIN_GRAY = "#d9d9d9"

def render(k, method):
    from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut
    n = len(X)
    if method == "K-Fold":
        cv = KFold(n_splits=k, shuffle=True, random_state=42)
    elif method == "Stratified K-Fold":
        cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    else:
        cv = LeaveOneOut()

    # Use the SAME splitter for the picture and the scoring, so they agree.
    splits = list(cv.split(X, y))
    n_folds = len(splits)
    show_k = min(n_folds, 10)
    ref_y = show_k + 0.6   # y-position of the "Overall" reference strip (gap above folds)

    # Batch points into 3 categories instead of one trace per point.
    train_x, train_y = [], []
    val0_x, val0_y = [], []   # held-out, class 0
    val1_x, val1_y = [], []   # held-out, class 1
    annotations = []

    for fi in range(show_k):
        tr_idx, va_idx = splits[fi]
        va_set = set(va_idx.tolist())
        for i in range(n):
            if i in va_set:
                if y[i] == 0:
                    val0_x.append(i); val0_y.append(fi)
                else:
                    val1_x.append(i); val1_y.append(fi)
            else:
                train_x.append(i); train_y.append(fi)
        # Per-fold validation class balance — the thing stratification protects.
        pos_rate = float(np.mean(y[va_idx])) if len(va_idx) else 0.0
        annotations.append(dict(
            x=n + 1, y=fi, xref="x", yref="y",
            text=f"{pos_rate*100:.0f}% class 1",
            showarrow=False, xanchor="left",
            font=dict(family=FONT['family'], size=11)))

    # --- Static reference strip: the whole dataset, colored by class ---
    overall = float(np.mean(y))
    ref0_x = [i for i in range(n) if y[i] == 0]
    ref1_x = [i for i in range(n) if y[i] == 1]
    annotations.append(dict(
        x=n + 1, y=ref_y, xref="x", yref="y",
        text=f"{overall*100:.0f}% class 1 (target)",
        showarrow=False, xanchor="left",
        font=dict(family=FONT['family'], size=11)))

    traces = [
        go.Scatter(x=train_x, y=train_y, mode="markers", name="Training",
            marker=dict(color=TRAIN_GRAY, size=8, opacity=0.7), hoverinfo="skip"),
        go.Scatter(x=val0_x, y=val0_y, mode="markers", name="Validation · class 0",
            marker=dict(color=PALETTE["secondary"], size=9, opacity=0.9), hoverinfo="skip"),
        go.Scatter(x=val1_x, y=val1_y, mode="markers", name="Validation · class 1",
            marker=dict(color=PALETTE["accent"], size=9, opacity=0.9), hoverinfo="skip"),
        # Reference strip (no gray — it's the full dataset, not a fold split)
        go.Scatter(x=ref0_x, y=[ref_y]*len(ref0_x), mode="markers", showlegend=False,
            marker=dict(color=PALETTE["secondary"], size=9, opacity=0.9), hoverinfo="skip"),
        go.Scatter(x=ref1_x, y=[ref_y]*len(ref1_x), mode="markers", showlegend=False,
            marker=dict(color=PALETTE["accent"], size=9, opacity=0.9), hoverinfo="skip"),
    ]

    scores = cross_val_score(DecisionTreeClassifier(random_state=42), X, y,
                             cv=cv, scoring="accuracy")
    title_k = n_folds
    layout = base_layout(
        title=f"{method} (k={title_k}) — Mean accuracy: {scores.mean():.3f} ± {scores.std():.3f}",
        xaxis_title="Sample index", yaxis_title="Fold")
    layout.update(height=360, annotations=annotations)
    layout.update(xaxis=dict(range=[-2, n + 16]))  # room for the right-side labels
    # Label the reference strip on the y-axis alongside the fold numbers
    layout.update(yaxis=dict(
        tickmode="array",
        tickvals=list(range(show_k)) + [ref_y],
        ticktext=[str(i) for i in range(show_k)] + ["Overall"],
        title="Fold"))

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces, layout=layout)))

    # --- Dynamic caption: explains the encoding + a method-aware note ---
    fold_rates = [float(np.mean(y[va_idx])) if len(va_idx) else 0.0
                  for _, va_idx in splits]
    spread = (max(fold_rates) - min(fold_rates)) if fold_rates else 0.0

    legend_line = (
        "<b>How to read this:</b> each row is one fold. "
        "<span style='color:#999'>Gray</span> = samples the model <b>trains</b> on; "
        "colored = the held-out samples it's <b>tested</b> on. "
        "Color shows each test point's class "
        f"(<span style='color:{PALETTE['secondary']}'>class 0</span> / "
        f"<span style='color:{PALETTE['accent']}'>class 1</span>). "
        "Every sample is held out for testing exactly once — read down a column. "
        "The top <b>Overall</b> strip shows the dataset's true class mix for comparison."
    )

    if method == "K-Fold":
        note = (f"<b>K-Fold</b> ignores the target when splitting, so each fold's test slice "
                f"lands at a different class-1 rate (here {min(fold_rates)*100:.0f}%–"
                f"{max(fold_rates)*100:.0f}%, vs {overall*100:.0f}% overall). "
                f"Spread across folds: {spread*100:.0f} points.")
    elif method == "Stratified K-Fold":
        note = (f"<b>Stratified K-Fold</b> looks at the target when splitting, forcing every "
                f"fold's test slice to ~{overall*100:.0f}% class 1. "
                f"Spread across folds: {spread*100:.0f} points — flat, by design.")
    else:
        note = ("<b>Leave-One-Out</b> holds out a single sample per fold, so each test set is "
                "100% one class by definition. That's why per-fold scores are all-or-nothing "
                f"and the ± spread is large (showing 10 of {n_folds} folds).")

    caption.value = (
        f"<div style='font-size:13px; line-height:1.5; padding:6px 2px'>"
        f"{legend_line}<br><br>{note}</div>"
    )

def on_change(change): render(k_slider.value, method_dd.value)
k_slider.observe(on_change, names="value")
method_dd.observe(on_change, names="value")
display(VBox([k_slider, method_dd, out, caption]))
render(5, "K-Fold")

---
## What's happening?

A teacher who tests students using the same exam questions every semester is measuring familiarity with those specific questions, not actual understanding. A fairer assessment uses different questions each time.

K-fold cross-validation works the same way:
1. Divide the dataset into k equal folds
2. Train on k-1 folds, validate on the remaining fold
3. Repeat k times, rotating which fold is held out
4. Average the k validation scores

The mean score is a better estimate of generalization performance. The standard deviation tells you how consistent the model is across different data subsets.

| Method | When to use | Computational cost | Risk |
|---|---|---|---|
| K-Fold | Most regression and balanced classification | Moderate (k × training time) | Can have class imbalance per fold |
| Stratified K-Fold | Classification with imbalanced classes | Same as K-Fold | None — handles imbalance |
| Leave-One-Out | Very small datasets (n < 50) | Very high (n × training time) | High variance in score estimates |

---
## Real-world example: Medical test accuracy

A diagnostic model trained on 200 patient records uses a single 80/20 split. By chance, 90% of positive cases land in the training set. The model looks good on the 20% validation set but fails when deployed to new hospitals.

With 5-fold stratified cross-validation, positive cases are evenly distributed across all folds. The resulting mean accuracy of 0.82 ± 0.04 is a far more honest estimate, and the 0.04 standard deviation signals that performance might vary by hospital population.

---
## Key takeaway

> **Cross-validation rotates which data is used for validation so every example is evaluated exactly once, giving a more reliable and less variable estimate of true model performance.**

---
*Next up: Loss Functions — how the model measures its own mistakes during training*